In [ ]:
!pip install osmnx geopandas pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 4.6 MB/s eta 0:00:00


In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd

# Core DFW Counties
dfw_counties = [
    "Dallas County, Texas", "Tarrant County, Texas", "Collin County, Texas",
    "Denton County, Texas", "Ellis County, Texas", "Johnson County, Texas",
    "Parker County, Texas", "Kaufman County, Texas", "Rockwall County, Texas",
    "Hunt County, Texas", "Wise County, Texas"
]

# Tags for large commercial/industrial assets
tags = {
    'building': ['commercial', 'industrial', 'warehouse', 'retail', 'supermarket', 'office']
}

print(f"Procedure initialized for {len(dfw_counties)} counties.")

Procedure initialized for 11 counties.


In [ ]:
all_county_data = []

for county in dfw_counties:
    try:
        print(f"--- Processing: {county} ---")
        # 1. Download buildings for the specific county
        gdf = ox.features_from_place(county, tags=tags)

        # 2. Keep only Polygons (ignore points/lines)
        gdf = gdf[gdf.geometry.type.isin(['Polygon', 'MultiPolygon'])]

        # 3. Project to Texas North Central (Feet) for accurate area math
        gdf_ft = gdf.to_crs(epsg=2276)
        gdf_ft['Building_Sq_Ft'] = gdf_ft.geometry.area

        # 4. Filter for > 20,000 sq ft
        large_gdf = gdf_ft[gdf_ft['Building_Sq_Ft'] > 20000].copy()

        # 5. Add a 'county_name' column so you know where each building is from
        large_gdf['county'] = county

        # 6. Select only the necessary columns to keep the file small
        # We ensure 'name' and 'building' type are included
        essential_cols = ['name', 'building', 'Building_Sq_Ft', 'county', 'geometry']
        available_cols = [c for c in essential_cols if c in large_gdf.columns]

        all_county_data.append(large_gdf[available_cols])
        print(f"Successfully added {len(large_gdf)} large buildings.")

    except Exception as e:
        print(f"Skipping {county}: No buildings found or error occurred.")

# Combine all counties into one Master DFW Dataset
dfw_master = pd.concat(all_county_data, ignore_index=True)
dfw_master_gdf = gpd.GeoDataFrame(dfw_master, crs=2276)

print(f"\nFinal Result: {len(dfw_master_gdf)} assets collected across the DFW Metroplex.")

--- Processing: Dallas County, Texas ---
Successfully added 5066 large buildings.
--- Processing: Tarrant County, Texas ---
Successfully added 1498 large buildings.
--- Processing: Collin County, Texas ---
Successfully added 1003 large buildings.
--- Processing: Denton County, Texas ---
Successfully added 738 large buildings.
--- Processing: Ellis County, Texas ---
Successfully added 86 large buildings.
--- Processing: Johnson County, Texas ---
Successfully added 66 large buildings.
--- Processing: Parker County, Texas ---
Successfully added 29 large buildings.
--- Processing: Kaufman County, Texas ---
Successfully added 29 large buildings.
--- Processing: Rockwall County, Texas ---
Successfully added 57 large buildings.
--- Processing: Hunt County, Texas ---
Successfully added 28 large buildings.
--- Processing: Wise County, Texas ---
Successfully added 12 large buildings.

Final Result: 8612 assets collected across the DFW Metroplex.


In [ ]:
# 1. Generate Latitude and Longitude for documentation
# Project back to GPS (4326) for the final coordinates
gps_centers = dfw_master_gdf.to_crs(epsg=4326).centroid
dfw_master_gdf['latitude'] = gps_centers.y
dfw_master_gdf['longitude'] = gps_centers.x

# 2. Save to CSV (dropping the complex geometry column)
filename = "DFW_Metroplex_Commercial_Assets_20k.csv"
dfw_master_gdf.drop(columns='geometry').to_csv(filename, index=False)

# 3. Download to your computer
from google.colab import files
files.download(filename)

print(f"Documentation saved as {filename}")

/tmp/ipykernel_1248/588752520.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gps_centers = dfw_master_gdf.to_crs(epsg=4326).centroid


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Documentation saved as DFW_Metroplex_Commercial_Assets_20k.csv
